# 01 · Pipeline de datos

Proyecto de titulación · Maestría en Inteligencia Artificial Aplicada (UDLA)
Amapola Technologies Corp.

Este notebook construye el dataset con grano de una fila por cliente y por mes, a partir del cruce
de los dos archivos exportados desde BigQuery y de las cuatro tablas del chatbot alojadas en MySQL,
las cuales se encuentran en la base `mia_capstone`.

El universo de estudio corresponde a los cliente-mes con `flujo = COBRANZA` según el archivo de
gestión, dado que dicho archivo define los clientes efectivamente gestionados. Cabe aclarar que el
flujo declarado en el archivo de pagos discrepa en 814 casos, motivo por el cual no se utiliza para
filtrar. Así también, se excluyen los documentos que no superan la validación ecuatoriana.

Se aplican tres reglas antes de cualquier cruce. En primer lugar, la cédula se normaliza con
`zfill(10)` y se valida en las tres fuentes. En segundo lugar, el mes se construye en formato
`YYYYMM`. Por último, la variable objetivo es `cumplimiento_conciliado`, cuyo desbalance se maneja
en la fase de modelado.

La salida es `outputs/dataset_clientemes_v1.csv`, anonimizado mediante SHA-256 con salt fijo.

## 0 · Configuración

In [ ]:
import os
import re
import hashlib
import secrets
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# --- Rutas -----------------------------------------------------------------
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_BASES = RAIZ / "bases"
DIR_OUTPUTS = RAIZ / "outputs"
DIR_MODELOS = RAIZ / "modelos"
DIR_OUTPUTS.mkdir(exist_ok=True)
DIR_MODELOS.mkdir(exist_ok=True)

CSV_GESTION = DIR_BASES / "registros_gestion.csv"
CSV_PAGOS = DIR_BASES / "registros_pagos.csv"
RUTA_DATASET = DIR_OUTPUTS / "dataset_clientemes_v1.csv"

# --- Ventana temporal del estudio (julio 2025 - abril 2026) ----------------
PERIODO_INICIO = "2025-07-01"   # inclusivo
PERIODO_FIN = "2026-05-01"      # exclusivo
MES_MIN, MES_MAX = "202507", "202604"

# --- Decisiones del pipeline -----------------------------------------------
# Universo del estudio: se filtra por el flujo declarado en el CSV de GESTION.
# El flujo del CSV de pagos discrepa en 814 cliente-mes y no se usa para filtrar.
FILTRO_FLUJO = "COBRANZA"

# respond describe al cliente en prod_db, no al evento conversacional, y no
# define el universo de cobranza. Aplicarlo baja la cobertura de 26.5% a 1.5%.
# La celda 3.4 mide ambos escenarios en cada ejecucion.
FILTRAR_RESPOND = False

# Campos posteriores al resultado del mes: fuga de informacion para el modelo.
INCLUIR_CAMPOS_POST_RESULTADO = False

# Excluir las cedulas que no superan la validacion ecuatoriana (incluye los
# dos pasaportes del universo, confirmados como tales).
EXCLUIR_CEDULAS_INVALIDAS = True

print("Raiz del proyecto:", RAIZ)
print("Gestion:", CSV_GESTION.exists(), "|", CSV_GESTION.name)
print("Pagos  :", CSV_PAGOS.exists(), "|", CSV_PAGOS.name)

### 0.1 · Conexión MySQL y mapeo del esquema

Las cuatro tablas están en `mia_capstone`. `ESQUEMA` mantiene el esquema por tabla como parámetro
para que, si `prod_db` o `prod_compromisos` vuelven a vivir en otra base, baste corregirlo aquí.

Credenciales por variable de entorno (los valores por defecto corresponden al MySQL local:
`root` sin contraseña):

```powershell
$env:MYSQL_USER = "root"; $env:MYSQL_PASSWORD = ""; $env:MYSQL_DATABASE = "mia_capstone"
```

In [ ]:
MYSQL = {
    "host": os.getenv("MYSQL_HOST", "127.0.0.1"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD", ""),
    "database": os.getenv("MYSQL_DATABASE", "mia_capstone"),
}

# Mapeo verificado contra la base local el 2026-08-09.
ESQUEMA = {
    # esquemas
    "bd_intents": "mia_capstone",
    "bd_clientes": "mia_capstone",
    # tablas
    "tabla_clientes_intents": "prod_clients_intents",
    "tabla_intents": "prod_intents",
    "tabla_clientes": "prod_db",
    "tabla_compromisos": "prod_compromisos",
    # mia_capstone.prod_clients_intents
    "ci_id_cliente": "id_cliente",
    "ci_id_intent": "id_intent",
    "ci_created_at": "created_at",
    # mia_capstone.prod_intents
    "i_id": "id",
    "i_nombre": "intent",
    # robocob.prod_db  (aqui viven cedula y respond)
    "c_id": "id",
    "c_cedula": "cedula",
    "c_respond": "respond",
    # robocob.prod_compromisos
    "cp_cedula": "cedula",
    "cp_fecha": "fecha",
    "cp_created_at": "created_at",
}

### 0.2 · Catálogo de intenciones

Nombres tomados del catálogo real (`mia_capstone.prod_intents`, 60 nombres distintos).
El chatbot registra el compromiso en dos pasos: `compromiso` (lo adquiere) y `compromiso_dia`
(confirma el plazo). **El plazo en días no está en el nombre de la intención**: se obtiene de
`robocob.prod_compromisos` como `fecha - created_at` (sección 3.5).

In [ ]:
# Intenciones de compromiso de pago
INTENTS_COMPROMISO = {"compromiso", "compromiso_dia"}

# Intención "ya pagué"
INTENTS_YA_PAGUE = {"ya_pague"}

# Red de seguridad por si aparecen nombres nuevos en el catálogo
PATRON_COMPROMISO = r"compromis|acuerdo[_\s-]?(?:de[_\s-]?)?pago|promesa[_\s-]?(?:de[_\s-]?)?pago"
PATRON_YA_PAGUE = r"ya[_\s-]?pagu|ya[_\s-]?cancel|pago[_\s-]?realizado"

def _sin_tildes(texto) -> str:
    return str(texto).translate(str.maketrans("áéíóúüÁÉÍÓÚÜñÑ", "aeiouuAEIOUUnN"))

def normalizar_intent(serie: pd.Series) -> pd.Series:
    """Minúsculas, sin tildes, sin espacios sobrantes."""
    return serie.astype("string").fillna("").map(_sin_tildes).str.lower().str.strip()

def es_compromiso(serie: pd.Series) -> pd.Series:
    return serie.isin(INTENTS_COMPROMISO) | serie.str.contains(PATRON_COMPROMISO, regex=True, na=False)

def es_ya_pague(serie: pd.Series) -> pd.Series:
    return serie.isin(INTENTS_YA_PAGUE) | serie.str.contains(PATRON_YA_PAGUE, regex=True, na=False)

### 0.3 · Normalización y validación de cédula

La cédula ecuatoriana consta de diez dígitos, de los cuales los dos primeros corresponden al código
de provincia, el tercero identifica el tipo de persona y el décimo es un dígito verificador
calculado por módulo 10.

`zfill(10)` recupera los ceros a la izquierda que se pierden cuando la cédula viaja como número, ya
que las provincias 01 a 09 empiezan con cero. Sin embargo, rellenar no garantiza que el resultado
sea un documento real, motivo por el cual se valida además de normalizar.

En este universo existen 47 documentos que no superan la validación, de los cuales 39 corresponden a
RUC de entidad, seis fallan el dígito verificador y dos resultaron ser pasaportes. Por ende, se
excluyen mediante la bandera `EXCLUIR_CEDULAS_INVALIDAS`.

In [ ]:
def normalizar_cedula(serie):
    # el replace del .0 es por las cedulas que BigQuery exporto como float,
    # si no se quita el zfill deja algo tipo 010094749.0 y no cruza con nada
    s = serie.astype("string").str.strip()
    s = s.str.replace(r"\.0+$", "", regex=True)
    s = s.str.replace(r"\D", "", regex=True)
    s = s.str.zfill(10)
    return s

def cedula_valida(c):
    if not isinstance(c, str) or len(c) != 10 or not c.isdigit():
        return False
    provincia = int(c[:2])
    if not (1 <= provincia <= 24 or provincia == 30):
        return False
    if int(c[2]) > 5:   # tercer digito 6-9 = RUC de entidad, no persona natural
        return False
    total = 0
    for i, d in enumerate(c[:9]):
        n = int(d) * (2 if i % 2 == 0 else 1)
        total += n - 9 if n > 9 else n
    return (10 - total % 10) % 10 == int(c[9])

def a_mes_yyyymm(serie_fecha):
    return pd.to_datetime(serie_fecha, errors="coerce").dt.strftime("%Y%m").astype("string")

def normalizar_mes(serie):
    s = serie.astype("string").str.strip().str.replace(r"\.0+$", "", regex=True)
    return s.str.replace(r"\D", "", regex=True)

def enmascarar(c):
    # se muestran enmascaradas porque el notebook se versiona: 0102030405 -> 01******05
    c = str(c)
    return c if len(c) < 5 else c[:2] + "*" * (len(c) - 4) + c[-2:]

def reporte_cedulas(df, nombre):
    # se llama en las tres fuentes para comparar longitudes antes de cruzar
    largos = df["cedula"].str.len().value_counts().sort_index().to_dict()
    unicas = pd.Series(df["cedula"].unique())
    validas = unicas.map(cedula_valida)
    print(f"[{nombre}] filas={len(df):,} | cedulas unicas={len(unicas):,} | longitudes={largos}")
    print(f"[{nombre}] validacion ecuatoriana: {int(validas.sum()):,} validas / "
          f"{int((~validas).sum()):,} invalidas")
    if (~validas).any():
        malas = unicas[~validas]
        print(f"[{nombre}]   provincias de las invalidas: {malas.str[:2].value_counts().head(5).to_dict()}")
        print(f"[{nombre}]   ejemplos: {[enmascarar(c) for c in malas.head(8)]}")

---
## 1 · Fuente 1: CSV de BigQuery

In [ ]:
gestion_raw = pd.read_csv(CSV_GESTION, dtype=str)
pagos_raw = pd.read_csv(CSV_PAGOS, dtype=str)

print("registros_gestion.csv:", gestion_raw.shape)
print(list(gestion_raw.columns))
print()
print("registros_pagos.csv  :", pagos_raw.shape)
print(list(pagos_raw.columns))
display(gestion_raw.head(3))
display(pagos_raw.head(3))

### 1.1 · Gestión a grano cliente-mes

Se toma el primer registro del mes para las variables predictoras y el último para el cierre. Cabe
aclarar que el export actual ya viene colapsado a cliente-mes, por lo que la agregación resulta
idempotente, aunque se mantiene porque seguiría siendo correcta si el export regresa al grano
cliente-día.

Se construyen dos objetos. Por un lado, `gestion_historico` conserva todos los flujos y sirve para
calcular `meses_consecutivos_mora`, ya que la mora es un hecho del cliente y un mes gestionado en
CONVENIO no debería romper la racha. Por otro lado, `gestion` corresponde al universo del estudio,
es decir, `flujo = COBRANZA` con cédula válida.

In [ ]:
COLS_PRIMER_REGISTRO = ["flujo", "dias_mora_inicial", "tipo_operacion", "bucket",
                        "mes_campania", "lote_control"]
COLS_ULTIMO_REGISTRO = ["dias_mora_cierre", "fecha_cierre_mes"]

g = gestion_raw.copy()
g["cedula_cruda"] = g["cedula"].astype("string").str.strip()
g["cedula"] = normalizar_cedula(g["cedula"])
g["mes"] = normalizar_mes(g["mes"])
g["flujo"] = g["flujo"].astype("string").str.strip().str.upper()
g["fecha_carpeta_gestion"] = pd.to_datetime(g["fecha_carpeta_gestion"], errors="coerce")

reporte_cedulas(g, "gestion")
print("filas duplicadas por (cedula, mes):", int(g.duplicated(["cedula", "mes"]).sum()))
print("\nflujo:", g["flujo"].value_counts(dropna=False).to_dict())

# Trazabilidad: qué cédulas venían con menos de 10 dígitos en el CSV
cortas = g.loc[g["cedula_cruda"].str.len() != 10, ["cedula_cruda", "cedula"]].drop_duplicates()
if len(cortas):
    print("\nDocumentos con longitud != 10 en el CSV de origen (pasaportes):")
    display(cortas.assign(valido_tras_zfill=cortas["cedula"].map(cedula_valida)))

g = g.drop(columns=["cedula_cruda"])
g = g.sort_values(["cedula", "mes", "fecha_carpeta_gestion"], na_position="last")
gr = g.groupby(["cedula", "mes"], as_index=False)

primeros = gr.first()[["cedula", "mes", "fecha_carpeta_gestion"] + COLS_PRIMER_REGISTRO]
ultimos = gr.last()[["cedula", "mes"] + COLS_ULTIMO_REGISTRO]
gestion_historico = primeros.merge(ultimos, on=["cedula", "mes"], how="inner")

for c in ["dias_mora_inicial", "dias_mora_cierre", "mes_campania"]:
    gestion_historico[c] = pd.to_numeric(gestion_historico[c], errors="coerce")
gestion_historico["fecha_cierre_mes"] = pd.to_datetime(gestion_historico["fecha_cierre_mes"],
                                                       errors="coerce")
for c in ["flujo", "tipo_operacion", "bucket", "lote_control"]:
    gestion_historico[c] = gestion_historico[c].astype("string").str.strip()
gestion_historico["cedula_es_valida"] = gestion_historico["cedula"].map(cedula_valida)

print("\ngestion_historico (todos los flujos):", gestion_historico.shape)

# --- Universo del estudio -------------------------------------------------
gestion = gestion_historico[gestion_historico["flujo"] == FILTRO_FLUJO].copy()
print(f"tras filtrar flujo == {FILTRO_FLUJO}: {gestion.shape}")

if EXCLUIR_CEDULAS_INVALIDAS:
    n = len(gestion)
    gestion = gestion[gestion["cedula_es_valida"]].copy()
    print(f"tras excluir documentos invalidos: {gestion.shape} "
          f"(-{n - len(gestion)} cliente-mes)")

gestion = gestion.drop(columns=["flujo"])   # constante tras el filtro
print(f"\nUniverso final: {len(gestion):,} cliente-mes | {gestion['cedula'].nunique():,} clientes")
display(gestion.head(3))

### 1.2 · Pagos → etiquetas

Pagos **no se filtra por su propio `flujo`**: el universo ya lo definió gestión. Se reporta la
discrepancia entre ambas etiquetas como control de calidad, nada más.

In [ ]:
p = pagos_raw.copy()
p["cedula"] = normalizar_cedula(p["cedula"])
p["mes"] = normalizar_mes(p["mes"])
p["flujo"] = p["flujo"].astype("string").str.strip().str.upper()
reporte_cedulas(p, "pagos")
print("filas duplicadas por (cedula, mes):", int(p.duplicated(["cedula", "mes"]).sum()))
print("\nflujo:", p["flujo"].value_counts(dropna=False).to_dict())

# Control de calidad: ¿coincide el flujo entre las dos fuentes?
comparacion = gestion_historico[["cedula", "mes", "flujo"]].merge(
    p[["cedula", "mes", "flujo"]], on=["cedula", "mes"], how="inner",
    suffixes=("_gestion", "_pagos"))
discrepantes = comparacion[comparacion["flujo_gestion"] != comparacion["flujo_pagos"]]
print(f"\nCliente-mes con flujo distinto entre gestion y pagos: {len(discrepantes):,}")
if len(discrepantes):
    print(pd.crosstab(discrepantes["flujo_gestion"], discrepantes["flujo_pagos"]).to_string())
    print("-> se respeta el flujo de gestion; pagos solo aporta las etiquetas")

p = p.drop(columns=["flujo"])

p["fecha_pago"] = pd.to_datetime(p["fecha_pago"], errors="coerce")
p["monto_recuperado"] = pd.to_numeric(p["monto_recuperado"], errors="coerce")
for c in ["cumplimiento_conciliado", "cumplimiento_ceroteo"]:
    p[c] = pd.to_numeric(p[c], errors="coerce").astype("Int64")

# Si hubiera más de una fila por cliente-mes: 1 si hubo al menos un pago confirmado
pagos = (
    p.groupby(["cedula", "mes"], as_index=False)
     .agg(
         cumplimiento_conciliado=("cumplimiento_conciliado", "max"),
         cumplimiento_ceroteo=("cumplimiento_ceroteo", "max"),
         fecha_pago=("fecha_pago", "min"),
         monto_recuperado=("monto_recuperado", "sum"),
     )
)

print("\npagos cliente-mes:", pagos.shape)
print("\ncumplimiento_conciliado:")
print(pagos["cumplimiento_conciliado"].value_counts(dropna=False).to_string())
print("\ncumplimiento_ceroteo:")
print(pagos["cumplimiento_ceroteo"].value_counts(dropna=False).to_string())

---
## 2 · Variable 10: meses consecutivos en mora

Meses **calendario consecutivos**, terminando en el mes actual, en los que el cliente inició el mes
con `dias_mora_inicial > 0`. Un mes sin mora, o un hueco en el calendario, reinicia el conteo.

Se calcula sobre `gestion_historico` (**todos los flujos**): un mes gestionado en CONVENIO sigue
siendo un mes en mora y no debe cortar la racha artificialmente.

In [ ]:
def calcular_meses_consecutivos_mora(df):
    # ojo: hay que contar meses de calendario consecutivos, no filas seguidas.
    # un cliente puede no aparecer un mes y eso corta la racha
    d = df[["cedula", "mes", "dias_mora_inicial"]].copy()
    d["periodo"] = pd.PeriodIndex(pd.to_datetime(d["mes"], format="%Y%m"), freq="M")
    d["en_mora"] = d["dias_mora_inicial"].fillna(0) > 0
    d = d.sort_values(["cedula", "periodo"])

    conteos = []
    for _, grupo in d.groupby("cedula", sort=False):
        racha, periodo_previo = 0, None
        for periodo, en_mora in zip(grupo["periodo"], grupo["en_mora"]):
            consecutivo = periodo_previo is not None and (periodo - periodo_previo).n == 1
            racha = (racha + 1 if consecutivo else 1) if en_mora else 0
            conteos.append(racha)
            periodo_previo = periodo

    d["meses_consecutivos_mora"] = conteos
    return d[["cedula", "mes", "meses_consecutivos_mora"]]


mora_consecutiva = calcular_meses_consecutivos_mora(gestion_historico)
print("calculada sobre", len(gestion_historico), "cliente-mes de todos los flujos")
print(mora_consecutiva["meses_consecutivos_mora"].value_counts().sort_index().to_string())

---
## 3 · Fuente 2: MySQL local (intenciones del chatbot)

In [ ]:
from sqlalchemy import create_engine, inspect, text
from urllib.parse import quote_plus

def crear_engine():
    url = (
        f"mysql+pymysql://{quote_plus(MYSQL['user'])}:{quote_plus(MYSQL['password'])}"
        f"@{MYSQL['host']}:{MYSQL['port']}/{MYSQL['database']}?charset=utf8mb4"
    )
    # connect_timeout evita que la celda se cuelgue indefinidamente si MySQL
    # acepta el socket pero no completa el handshake.
    return create_engine(url, pool_pre_ping=True,
                         connect_args={"connect_timeout": 10, "read_timeout": 300})

engine = crear_engine()
with engine.connect() as con:
    print("Conectado a:", con.execute(text("SELECT DATABASE(), VERSION()")).fetchone())

### 3.1 · Introspección de los dos esquemas

Verifica que existan las cuatro tablas y las columnas mapeadas en `ESQUEMA`, en `mia_capstone`
y en `robocob`. Si algo no coincide, se corrige en la celda 0.1 y se reejecuta desde ahí.

In [ ]:
E = ESQUEMA
TABLAS = {
    (E["bd_intents"], E["tabla_clientes_intents"]):
        [E["ci_id_cliente"], E["ci_id_intent"], E["ci_created_at"]],
    (E["bd_intents"], E["tabla_intents"]):
        [E["i_id"], E["i_nombre"]],
    (E["bd_clientes"], E["tabla_clientes"]):
        [E["c_id"], E["c_cedula"], E["c_respond"]],
    (E["bd_clientes"], E["tabla_compromisos"]):
        [E["cp_cedula"], E["cp_fecha"], E["cp_created_at"]],
}

insp = inspect(engine)
for (esquema, tabla), columnas in TABLAS.items():
    existentes = set(insp.get_table_names(schema=esquema))
    if tabla not in existentes:
        print(f"[FALTA] {esquema}.{tabla} no existe. Tablas disponibles: {sorted(existentes)[:12]}\n")
        continue
    reales = [c["name"] for c in insp.get_columns(tabla, schema=esquema)]
    faltantes = [c for c in columnas if c not in reales]
    estado = "OK" if not faltantes else "REVISAR"
    with engine.connect() as con:
        n = con.execute(text(f"SELECT COUNT(*) FROM `{esquema}`.`{tabla}`")).scalar()
    print(f"[{estado}] {esquema}.{tabla}  ({n:,} filas)")
    print(f"        columnas: {reales}")
    if faltantes:
        print(f"        FALTAN: {faltantes}")
    print()

### 3.2 · Extracción de eventos conversacionales

JOIN entre esquemas: `mia_capstone.prod_clients_intents` ⋈ `mia_capstone.prod_intents`
⋈ `robocob.prod_db`. La cédula sale de `prod_db`, igual que el `respond`.

Se extrae **sin** el filtro `respond` y se marca cada evento con su valor, para poder medir en la
celda 3.4 cuánto cuesta aplicarlo antes de decidir.

In [ ]:
SQL_INTENCIONES = f"""
SELECT
    ci.{E['ci_id_cliente']}                          AS id_cliente,
    LPAD(TRIM(c.{E['c_cedula']}), 10, '0')           AS cedula,
    c.{E['c_respond']}                               AS respond,
    i.{E['i_nombre']}                                AS intent,
    ci.{E['ci_created_at']}                          AS created_at
FROM `{E['bd_intents']}`.`{E['tabla_clientes_intents']}` ci
JOIN `{E['bd_intents']}`.`{E['tabla_intents']}`  i ON i.{E['i_id']} = ci.{E['ci_id_intent']}
JOIN `{E['bd_clientes']}`.`{E['tabla_clientes']}` c ON c.{E['c_id']} = ci.{E['ci_id_cliente']}
WHERE ci.{E['ci_created_at']} >= :inicio
  AND ci.{E['ci_created_at']} <  :fin
"""
print(SQL_INTENCIONES)

intents_raw = pd.read_sql(text(SQL_INTENCIONES), engine,
                          params={"inicio": PERIODO_INICIO, "fin": PERIODO_FIN})
print("Eventos extraidos:", f"{len(intents_raw):,}")
print("Clientes          :", f"{intents_raw['id_cliente'].nunique():,}")
print("Cedulas           :", f"{intents_raw['cedula'].nunique():,}")
display(intents_raw.head())

# Respaldo local (contiene cédulas: queda en bases/, excluido del repositorio)
try:
    intents_raw.to_parquet(DIR_BASES / "intents_raw.parquet", index=False)
    print("\nRespaldo:", DIR_BASES / "intents_raw.parquet")
except Exception as e:
    intents_raw.to_csv(DIR_BASES / "intents_raw.csv", index=False, encoding="utf-8")
    print("\nRespaldo en CSV (parquet no disponible:", e, ")")

### 3.3 · Eventos que no se pueden cruzar

`prod_db` es la tabla que traduce `id_cliente` a `cedula`, de manera que los eventos cuyo
identificador no figura en ella no tienen cédula recuperable y quedan fuera del cruce.

- Con la fotografía inicial, que contenía 10.036 clientes, el 56,6 % de los eventos quedaba sin
  traducción.
- Con la carga histórica de 18.635 clientes, cuyos identificadores cubren el rango completo,
  dicha proporción baja al 18,7 %.

In [ ]:
SQL_HUERFANOS = f"""
SELECT COUNT(*) AS eventos, COUNT(DISTINCT ci.{E['ci_id_cliente']}) AS clientes
FROM `{E['bd_intents']}`.`{E['tabla_clientes_intents']}` ci
LEFT JOIN `{E['bd_clientes']}`.`{E['tabla_clientes']}` c ON c.{E['c_id']} = ci.{E['ci_id_cliente']}
WHERE c.{E['c_id']} IS NULL
"""
huerfanos = pd.read_sql(text(SQL_HUERFANOS), engine)
totales = pd.read_sql(
    text(f"SELECT COUNT(*) AS eventos, COUNT(DISTINCT {E['ci_id_cliente']}) AS clientes "
         f"FROM `{E['bd_intents']}`.`{E['tabla_clientes_intents']}`"), engine)

print("Histórico completo de prod_clients_intents:")
print(f"  eventos : {totales.eventos[0]:,}  |  clientes: {totales.clientes[0]:,}")
print("Sin match en robocob.prod_db (cédula irrecuperable):")
print(f"  eventos : {huerfanos.eventos[0]:,} ({huerfanos.eventos[0]/totales.eventos[0]:.1%})"
      f"  |  clientes: {huerfanos.clientes[0]:,} ({huerfanos.clientes[0]/totales.clientes[0]:.1%})")

### 3.4 · Impacto del filtro `respond = 2`

Se resolvió no aplicar este filtro, por dos motivos:

- `respond` es un atributo del cliente dentro de `prod_db`, no del evento conversacional, de modo que
  no describe lo que ocurrió en la conversación sino el estado del cliente en la campaña.
- Dicho filtro tampoco define el universo de cobranza, el cual queda delimitado por
  `flujo = COBRANZA` en el archivo de gestión.

Por ende se deja `FILTRAR_RESPOND = False`. Cabe aclarar que la celda mide ambos escenarios en cada
ejecución, con el fin de dejar constancia del criterio aplicado.

In [ ]:
cedulas_universo = set(gestion["cedula"])
pares_universo = set(zip(gestion["cedula"], gestion["mes"]))

def medir_cobertura(df, etiqueta):
    ced = set(df["cedula"])
    cruzan = ced & cedulas_universo
    # La cobertura se mide sobre el par (cedula, mes): que la cedula exista en el
    # universo no basta, el mes tambien tiene que haber sido gestionado.
    mes = a_mes_yyyymm(df["created_at"])
    pares = set(zip(df["cedula"], mes)) & pares_universo
    print(f"{etiqueta}")
    print(f"   eventos                       : {len(df):,}")
    print(f"   cedulas                       : {len(ced):,}")
    print(f"   cedulas que cruzan con gestion: {len(cruzan):,} "
          f"({len(cruzan)/len(cedulas_universo):.1%} del universo de {len(cedulas_universo):,})")
    print(f"   cliente-mes con conversacion  : {len(pares):,} "
          f"({len(pares)/len(gestion):.1%} de {len(gestion):,})")
    print()

medir_cobertura(intents_raw, "SIN filtro respond")
medir_cobertura(intents_raw[intents_raw["respond"] == 2], "CON respond = 2")

print(f"FILTRAR_RESPOND = {FILTRAR_RESPOND} (celda 0)")
if FILTRAR_RESPOND:
    intents_raw = intents_raw[intents_raw["respond"] == 2].copy()
    print("-> se aplica el filtro respond = 2")
else:
    print("-> se conservan todos los eventos")

### 3.5 · Variable 4: plazo del compromiso

El nombre de la intención no lleva el plazo: `compromiso_dia` solo indica que el cliente eligió uno.
El valor está en `robocob.prod_compromisos`, como la diferencia entre la fecha comprometida y la
fecha en que se registró el compromiso.

In [ ]:
SQL_COMPROMISOS = f"""
SELECT
    cp.{E['cp_cedula']}                                 AS cedula,
    cp.{E['cp_created_at']}                             AS created_at,
    DATEDIFF(cp.{E['cp_fecha']}, DATE(cp.{E['cp_created_at']})) AS dias_compromiso
FROM `{E['bd_clientes']}`.`{E['tabla_compromisos']}` cp
WHERE cp.{E['cp_created_at']} >= :inicio
  AND cp.{E['cp_created_at']} <  :fin
"""
compromisos_raw = pd.read_sql(text(SQL_COMPROMISOS), engine,
                              params={"inicio": PERIODO_INICIO, "fin": PERIODO_FIN})

# Reglas 1 y 2 aplicadas en pandas, igual que en las otras dos fuentes
compromisos_raw["cedula"] = normalizar_cedula(compromisos_raw["cedula"])
compromisos_raw["created_at"] = pd.to_datetime(compromisos_raw["created_at"], errors="coerce")
compromisos_raw["mes"] = a_mes_yyyymm(compromisos_raw["created_at"])

print("Compromisos en la ventana:", len(compromisos_raw))
print("\nDistribucion del plazo elegido (dias):")
print(compromisos_raw["dias_compromiso"].value_counts().sort_index().to_string())
print("\nCompromisos por mes:")
print(compromisos_raw.groupby("mes").size().to_string())

# Un compromiso por cliente-mes: el primero del mes
compromisos = (
    compromisos_raw.sort_values(["cedula", "mes", "created_at"])
                   .groupby(["cedula", "mes"], as_index=False)
                   .first()[["cedula", "mes", "dias_compromiso"]]
)
compromisos["dias_compromiso"] = pd.to_numeric(compromisos["dias_compromiso"],
                                               errors="coerce").astype("Int64")
print("\ncompromisos cliente-mes:", compromisos.shape)

---
## 4 · Variables conversacionales por cliente-mes

In [ ]:
ints = intents_raw.copy()
ints["cedula"] = normalizar_cedula(ints["cedula"])
ints["created_at"] = pd.to_datetime(ints["created_at"], errors="coerce")
ints["intent"] = normalizar_intent(ints["intent"])

antes = len(ints)
ints = ints.dropna(subset=["created_at"])
ints = ints[ints["cedula"].str.len() == 10]
ints = ints[ints["intent"] != ""]          # el catálogo tiene 2 intents sin nombre
print(f"Descartados {antes - len(ints):,} eventos sin fecha, sin nombre de intent o con cedula invalida")

ints["mes"] = a_mes_yyyymm(ints["created_at"])
ints = ints[ints["mes"].between(MES_MIN, MES_MAX)]

reporte_cedulas(ints, "intenciones")
print("\nEventos por mes:")
print(ints.groupby("mes").agg(eventos=("intent", "size"), cedulas=("cedula", "nunique")).to_string())

### 4.1 · Catálogo real de intenciones

Si aparece alguna intención de compromiso o de pago que no quede marcada, se agrega a
`INTENTS_COMPROMISO` / `INTENTS_YA_PAGUE` en la celda 0.2.

In [ ]:
catalogo = (
    ints.groupby("intent")
        .agg(eventos=("intent", "size"), clientes=("cedula", "nunique"))
        .sort_values("eventos", ascending=False)
)
serie_nombres = pd.Series(catalogo.index, index=catalogo.index).astype("string")
catalogo["es_compromiso"] = es_compromiso(serie_nombres)
catalogo["es_ya_pague"] = es_ya_pague(serie_nombres)

print("Intenciones distintas:", len(catalogo))
display(catalogo.head(40))
print("\nMarcadas COMPROMISO:", catalogo[catalogo["es_compromiso"]].index.tolist())
print("Marcadas YA PAGUE  :", catalogo[catalogo["es_ya_pague"]].index.tolist())

### 4.2 · Agregación (variables 1, 2, 3, 5, 6, 7 y 8)

| # | Variable | Definición |
|---|---|---|
| 1 | `total_intenciones` | Intenciones activadas en el mes |
| 2 | `intenciones_distintas` | Intenciones únicas en el mes |
| 3 | `activo_compromiso` | 1 si activó alguna intención de compromiso de pago |
| 5 | `activo_ya_pague` | 1 si activó la intención `ya_pague` |
| 6 | `ultima_intencion_antes_compromiso` | Intención inmediatamente previa al primer compromiso del mes |
| 7 | `hora_primera_interaccion` | Hora (0-23) de la primera interacción del mes |
| 8 | `dia_semana_primera_interaccion` | Día de la semana (0 = lunes … 6 = domingo) |

La variable 4 (`dias_compromiso`) llega desde `prod_compromisos` en la sección 5.

In [ ]:
SIN_COMPROMISO = "NO_APLICA"       # no activó compromiso ese mes
COMPROMISO_PRIMERO = "SIN_PREVIA"  # el compromiso fue la primera intención del mes
SIN_CONVERSACION = "SIN_DATOS"     # cliente-mes sin registros en el chatbot

def resumir_cliente_mes(g):
    # el sort es imprescindible: la intencion previa al compromiso depende del orden
    g = g.sort_values("created_at")
    intents = g["intent"].to_numpy()
    compromiso = es_compromiso(g["intent"]).to_numpy()
    primera = g["created_at"].iloc[0]

    if compromiso.any():
        pos = int(np.argmax(compromiso))
        previa = intents[pos - 1] if pos > 0 else COMPROMISO_PRIMERO
    else:
        previa = SIN_COMPROMISO

    return pd.Series({
        "total_intenciones": len(g),
        "intenciones_distintas": g["intent"].nunique(),
        "activo_compromiso": int(compromiso.any()),
        "activo_ya_pague": int(es_ya_pague(g["intent"]).any()),
        "ultima_intencion_antes_compromiso": previa,
        "hora_primera_interaccion": int(primera.hour),
        "dia_semana_primera_interaccion": int(primera.dayofweek),
    })


features_intents = (
    ints.groupby(["cedula", "mes"], sort=False)
        .apply(resumir_cliente_mes)
        .reset_index()
)
for c in ["total_intenciones", "intenciones_distintas", "activo_compromiso",
          "activo_ya_pague", "hora_primera_interaccion", "dia_semana_primera_interaccion"]:
    features_intents[c] = features_intents[c].astype("Int64")

print("features conversacionales:", features_intents.shape)
print("cliente-mes con compromiso:", int(features_intents["activo_compromiso"].sum()))
print("cliente-mes con ya_pague  :", int(features_intents["activo_ya_pague"].sum()))
display(features_intents.head())

---
## 5 · Cruce de las tres fuentes

El universo lo define **gestión** (todo cliente-mes gestionado). Pagos, historial de mora,
intenciones y compromisos se unen por la izquierda; un cliente-mes sin conversación registrada
conserva su fila con los contadores en 0.

In [ ]:
dataset = gestion.merge(pagos, on=["cedula", "mes"], how="left", validate="one_to_one")
print("tras unir pagos      :", dataset.shape)

dataset = dataset.merge(mora_consecutiva, on=["cedula", "mes"], how="left", validate="one_to_one")
print("tras unir mora hist. :", dataset.shape)

dataset = dataset.merge(features_intents, on=["cedula", "mes"], how="left", validate="one_to_one")
print("tras unir intenciones:", dataset.shape)

dataset = dataset.merge(compromisos, on=["cedula", "mes"], how="left", validate="one_to_one")
print("tras unir compromisos:", dataset.shape)

cobertura = dataset["total_intenciones"].notna().mean()
print(f"\nCobertura conversacional: {cobertura:.1%} de los cliente-mes tienen datos del chatbot")
print(f"Cobertura de plazo (var. 4): {dataset['dias_compromiso'].notna().mean():.1%}")

# Relleno explícito de los cliente-mes sin conversación
dataset["tiene_conversacion"] = dataset["total_intenciones"].notna().astype("Int64")
for c in ["total_intenciones", "intenciones_distintas", "activo_compromiso", "activo_ya_pague"]:
    dataset[c] = dataset[c].fillna(0).astype("Int64")
dataset["ultima_intencion_antes_compromiso"] = (
    dataset["ultima_intencion_antes_compromiso"].fillna(SIN_CONVERSACION).astype("string")
)
# hora, día de semana y dias_compromiso quedan nulos: XGBoost maneja NaN de forma nativa

sin_etiqueta = int(dataset["cumplimiento_conciliado"].isna().sum())
print(f"\nCliente-mes sin fila en pagos: {sin_etiqueta}")
if sin_etiqueta:
    for c in ["cumplimiento_conciliado", "cumplimiento_ceroteo"]:
        dataset[c] = dataset[c].fillna(0).astype("Int64")
    print("  -> imputados a 0 (sin registro de pago = no cumplió)")

---
## 6 · Validación del dataset

In [ ]:
PREDICTORAS = [
    "total_intenciones",                    # 1
    "intenciones_distintas",                # 2
    "activo_compromiso",                    # 3
    "dias_compromiso",                      # 4
    "activo_ya_pague",                      # 5
    "ultima_intencion_antes_compromiso",    # 6
    "hora_primera_interaccion",             # 7
    "dia_semana_primera_interaccion",       # 8
    "dias_mora_inicial",                    # 9
    "meses_consecutivos_mora",              # 10
]
CONTEXTO_CARTERA = ["tipo_operacion", "bucket", "mes_campania", "lote_control", "tiene_conversacion"]
if not EXCLUIR_CEDULAS_INVALIDAS:
    CONTEXTO_CARTERA.append("cedula_es_valida")
TARGETS = ["cumplimiento_conciliado", "cumplimiento_ceroteo"]

print("=" * 70)
print("CONTEO DE REGISTROS")
print("=" * 70)
print(f"Universo            : flujo == {FILTRO_FLUJO} (segun CSV de gestion)")
print(f"Filas (cliente-mes) : {len(dataset):,}")
print(f"Clientes unicos     : {dataset['cedula'].nunique():,}")
print(f"Meses               : {dataset['mes'].nunique()} ({dataset['mes'].min()} - {dataset['mes'].max()})")
print(f"Duplicados (ced,mes): {int(dataset.duplicated(['cedula', 'mes']).sum())}")
print(f"Documentos invalidos: {int((~dataset['cedula_es_valida']).sum())}")

print("\nRegistros y cobertura conversacional por mes:")
print(dataset.groupby("mes").agg(
    registros=("cedula", "size"),
    con_chatbot=("tiene_conversacion", "sum"),
    tasa_chatbot=("tiene_conversacion", "mean"),
).to_string())

In [ ]:
print("=" * 70)
print("DISTRIBUCION DEL TARGET")
print("=" * 70)
for t in TARGETS:
    conteo = dataset[t].value_counts().sort_index()
    positivos, negativos = int(conteo.get(1, 0)), int(conteo.get(0, 0))
    ratio = negativos / positivos if positivos else float("nan")
    print(f"\n{t}")
    print(f"  0 = {negativos:,} | 1 = {positivos:,}")
    print(f"  tasa de positivos = {positivos/len(dataset):.2%}  |  desbalance neg:pos = {ratio:.1f} : 1")
    print(f"  scale_pos_weight sugerido = {ratio:.3f}")

print("\nConcordancia conciliado vs ceroteo:")
print(pd.crosstab(dataset["cumplimiento_conciliado"], dataset["cumplimiento_ceroteo"],
                  rownames=["conciliado"], colnames=["ceroteo"]).to_string())

print("\nTasa de cumplimiento_conciliado por mes:")
print(dataset.groupby("mes")["cumplimiento_conciliado"].agg(["size", "sum", "mean"]).to_string())

print("\nTasa de cumplimiento segun contacto con el chatbot:")
print(dataset.groupby("tiene_conversacion")["cumplimiento_conciliado"].agg(["size", "sum", "mean"]).to_string())

In [ ]:
print("=" * 70)
print("NULOS Y TIPOS")
print("=" * 70)
resumen = pd.DataFrame({
    "tipo": dataset.dtypes.astype(str),
    "nulos": dataset.isna().sum(),
    "%_nulos": (dataset.isna().mean() * 100).round(2),
    "unicos": dataset.nunique(),
})
display(resumen.sort_values("%_nulos", ascending=False))

print("\nNulos en las 10 predictoras:")
print(dataset[PREDICTORAS].isna().sum().to_string())
print("\nNota: dias_compromiso es nulo cuando no hay compromiso registrado en prod_compromisos;")
print("hora y dia de semana son nulos cuando el cliente-mes no tuvo conversacion.")

In [ ]:
print("=" * 70)
print("COHERENCIA")
print("=" * 70)
chequeos = {
    "dias_compromiso fuera de {1,3,5}":
        int((~dataset["dias_compromiso"].isin([1, 3, 5]) & dataset["dias_compromiso"].notna()).sum()),
    "dias_compromiso <= 0":
        int((dataset["dias_compromiso"] <= 0).sum()),
    "hora fuera de 0-23":
        int((~dataset["hora_primera_interaccion"].between(0, 23) & dataset["hora_primera_interaccion"].notna()).sum()),
    "dia_semana fuera de 0-6":
        int((~dataset["dia_semana_primera_interaccion"].between(0, 6) & dataset["dia_semana_primera_interaccion"].notna()).sum()),
    "intenciones_distintas > total_intenciones":
        int((dataset["intenciones_distintas"] > dataset["total_intenciones"]).sum()),
    "dias_mora_inicial negativo":
        int((dataset["dias_mora_inicial"] < 0).sum()),
    "mes fuera de la ventana del estudio":
        int((~dataset["mes"].between(MES_MIN, MES_MAX)).sum()),
    "compromiso registrado sin intent de compromiso":
        int((dataset["dias_compromiso"].notna() & (dataset["activo_compromiso"] == 0)).sum()),
}
for nombre, n in chequeos.items():
    print(f"  [{'OK ' if n == 0 else 'REV'}] {nombre}: {n}")

print("\nDos hallazgos esperados, no son errores del pipeline:")
print("  - Plazos de 2 dias: existen en prod_compromisos (70 registros en total). El chatbot")
print("    ofrece 1/3/5, pero la tabla guarda la fecha real comprometida y algunos caen en 2.")
print("  - Compromisos sin intent de compromiso en el mismo cliente-mes: ~18.7% de los eventos")
print("    conversacionales no tienen cedula recuperable en prod_db, asi que el compromiso queda")
print("    registrado pero su rastro conversacional no se pudo cruzar.")

numericas = [c for c in PREDICTORAS
             if dataset[c].dtype.kind in "if" or str(dataset[c].dtype).startswith("Int")]
print("\nEstadisticos de las predictoras numericas:")
display(dataset[numericas].astype("float").describe().T)

---
## 7 · Anonimización y exportación

1. `cedula` → `id_anonimo` = SHA-256 de `salt + cedula` (salt fijo, fuera del repositorio).
2. Se eliminan todos los campos de identificación directa.
3. Se audita el dataset exportado en busca de datos personales identificables; si aparece alguno,
   la exportación se aborta.

In [ ]:
def obtener_salt() -> str:
    salt = os.getenv("CAPSTONE_SALT")
    if salt:
        print("Salt tomado de la variable de entorno CAPSTONE_SALT")
        return salt
    ruta = RAIZ / ".salt"
    if ruta.exists():
        print(f"Salt tomado de {ruta}")
        return ruta.read_text(encoding="utf-8").strip()
    salt = secrets.token_hex(32)
    ruta.write_text(salt, encoding="utf-8")
    print(f"Se genero un salt nuevo en {ruta}. CONSERVARLO: sin el, los hashes no son reproducibles.")
    return salt


def hashear(serie, salt):
    return serie.astype("string").map(
        lambda v: hashlib.sha256((salt + v).encode("utf-8")).hexdigest())
# print(hashear(pd.Series(["0102030405"]), SALT))   # para comprobar contra el prototipo


SALT = obtener_salt()
dataset["id_anonimo"] = hashear(dataset["cedula"], SALT)

print("\ncedulas unicas :", dataset["cedula"].nunique())
print("hashes unicos  :", dataset["id_anonimo"].nunique(), "(deben coincidir: sin colisiones)")
print("ejemplo hash   :", dataset["id_anonimo"].iloc[0])

In [ ]:
PATRON_COLUMNA_PII = (
    r"cedul|identific|documento|nombre|apellid|raz[oó]n[_\s]?social|"
    r"tarjet|card|pan\b|cuenta|iban|"
    r"direcc|domicilio|ciudad|barrio|"
    r"correo|e?mail|"
    r"celular|tel[eé]fon|phone|movil|contacto|"
    r"id_cliente|usuario|user"
)

columnas_pii = [c for c in dataset.columns if re.search(PATRON_COLUMNA_PII, c, flags=re.IGNORECASE)]
print("Columnas descartadas por identificacion directa:", columnas_pii)

columnas_finales = ["id_anonimo", "mes"] + PREDICTORAS + CONTEXTO_CARTERA + TARGETS
if INCLUIR_CAMPOS_POST_RESULTADO:
    columnas_finales += ["dias_mora_cierre", "fecha_pago", "monto_recuperado"]
    print("AVISO: se incluyen campos posteriores al resultado del mes (fuga de informacion "
          "si se usan como predictores).")
else:
    print("Excluidos por fuga de informacion: dias_mora_cierre, fecha_pago, monto_recuperado, "
          "fecha_cierre_mes, fecha_carpeta_gestion")

# 'cedula_es_valida' es una marca de calidad, no un identificador, pero su nombre
# coincide con el patron PII: se renombra a 'registro_valido'.
columnas_finales = [c for c in columnas_finales if c in dataset.columns]
dataset_final = dataset[columnas_finales].copy()
dataset_final = dataset_final.rename(columns={"cedula_es_valida": "registro_valido"})

sobrantes = [c for c in dataset_final.columns
             if re.search(PATRON_COLUMNA_PII, c, flags=re.IGNORECASE)]
assert not sobrantes, f"Quedaron columnas identificatorias: {sobrantes}"
print("\nColumnas del dataset final:", list(dataset_final.columns))

In [ ]:
def auditar_pii(df: pd.DataFrame) -> bool:
    """Recorre los valores del dataset buscando patrones de dato personal."""
    patrones = {
        "correo electronico": r"[\w.+-]+@[\w-]+\.[\w.]+",
        "cedula (10 digitos)": r"^\d{10}$",
        "tarjeta (13-19 digitos)": r"^\d{13,19}$",
        "celular ecuatoriano": r"^(?:593)?0?9\d{8}$",
    }
    hallazgos = []
    for col in df.columns:
        if col == "id_anonimo":
            continue
        valores = df[col].dropna().astype(str)
        if valores.empty:
            continue
        muestra = valores.drop_duplicates().head(5000)
        for nombre, patron in patrones.items():
            coinciden = muestra[muestra.str.match(patron)]
            if len(coinciden):
                hallazgos.append((col, nombre, len(coinciden), coinciden.iloc[0]))
    if hallazgos:
        print("SE ENCONTRARON POSIBLES DATOS PERSONALES:")
        for col, nombre, n, ejemplo in hallazgos:
            print(f"  - {col}: {n} valores parecidos a {nombre} (ej. {ejemplo})")
        return False
    print("Auditoria PII: ningun valor coincide con patrones de dato personal identificable.")
    return True


limpio = auditar_pii(dataset_final)
print("id_anonimo -> SHA-256 de 64 caracteres:",
      bool(dataset_final["id_anonimo"].str.fullmatch(r"[0-9a-f]{64}").all()))
assert limpio, "El dataset contiene datos personales: revisar antes de exportar."

In [ ]:
dataset_final.to_csv(RUTA_DATASET, index=False, encoding="utf-8")

diccionario = pd.DataFrame({
    "columna": dataset_final.columns,
    "tipo": [str(dataset_final[c].dtype) for c in dataset_final.columns],
    "nulos": [int(dataset_final[c].isna().sum()) for c in dataset_final.columns],
    "unicos": [int(dataset_final[c].nunique()) for c in dataset_final.columns],
    "rol": [
        "identificador" if c in ("id_anonimo", "mes")
        else "target" if c in TARGETS
        else "predictora" if c in PREDICTORAS
        else "contexto"
        for c in dataset_final.columns
    ],
})
diccionario.to_csv(DIR_OUTPUTS / "diccionario_dataset_v1.csv", index=False, encoding="utf-8")

print(f"Exportado: {RUTA_DATASET}  ({len(dataset_final):,} filas x {dataset_final.shape[1]} columnas)")
print(f"Exportado: {DIR_OUTPUTS / 'diccionario_dataset_v1.csv'}")
display(diccionario)
display(dataset_final.head())

---
### Siguiente fase

`02_eda.ipynb`, que contiene el análisis exploratorio sobre `outputs/dataset_clientemes_v1.csv`.